# GET vs POST in RESTful Routing

In RESTful routing, the core difference is that **GET** retrieves data from a server, while **POST** sends data to a server to create a new resource.

## Quick Comparison

| Feature | GET Request | POST Request |
|---|---|---|
| CRUD Operation | Read | Create |
| Data Location | URL query parameters | Request body |
| Safe | Yes — does not alter state | No — alters state |
| Idempotent | Yes | No — repeats create duplicates |
| Caching & History | Cacheable, saved in browser history | Not cached, not in history |
| Data Restrictions | ASCII text, practical length limit (~2000 chars in browsers) | No size limit, supports binary and multipart |
| Bookmarkable | Yes | No |
| Typical Success Status | `200 OK` | `201 Created` |
| Security | Data visible in URL, logs, and Referer headers | Data hidden in body — but see caveat below |

## Core Mechanics

### 1. GET Requests (Read)

A GET request fetches a representation of a resource. It must remain **safe** and **idempotent** — calling it once or a hundred times yields the same result without changing server state.

- **Collection route:** `GET /api/v1/products` retrieves a list of all products.
- **Member route:** `GET /api/v1/products/42` retrieves the product with ID 42.
- **Data transmission:** Parameters are appended to the URL string, e.g. `/products?category=electronics&sort=price`. Never use GET for sensitive data like passwords.

```js
// Express — collection with query params
app.get('/api/v1/products', (req, res) => {
  const { category, sort } = req.query;   // from the URL string
  res.json(findProducts({ category, sort }));
});

// Express — member route with a path param
app.get('/api/v1/products/:id', (req, res) => {
  const product = findById(req.params.id);
  if (!product) return res.status(404).json({ error: 'Not found' });
  res.json(product);
});
```

### 2. POST Requests (Create)

A POST request submits data to a resource, causing a state change or creating a new record. It is neither safe nor idempotent — sending the identical request repeatedly keeps creating duplicates.

- **Target route:** `POST /api/v1/products` submits data to the collection to instantiate a new item.
- **Data transmission:** Data travels in the HTTP request body, not the URL. Supports JSON, files, and multipart data.
- **Response:** A successful execution typically returns `201 Created` plus the newly generated object, often with a `Location` header pointing at the new resource.

```js
app.use(express.json());                          // parses application/json
app.use(express.urlencoded({ extended: true }));  // parses HTML form posts

app.post('/api/v1/products', (req, res) => {
  const product = createProduct(req.body);        // from the request body
  res.status(201)
     .location(`/api/v1/products/${product.id}`)
     .json(product);
});
```

---

## Additions Worth Knowing

### The full verb set

GET and POST are two of five. The rest complete CRUD:

| Verb | Purpose | Idempotent | Typical Route |
|---|---|---|---|
| GET | Read | Yes | `/products` or `/products/42` |
| POST | Create | No | `/products` |
| PUT | Replace entire resource | Yes | `/products/42` |
| PATCH | Update part of a resource | No (in principle) | `/products/42` |
| DELETE | Remove | Yes | `/products/42` |

**PUT vs PATCH:** PUT sends the complete replacement object — omitted fields are wiped. PATCH sends only the fields that change. PUT is idempotent because sending the same full object twice leaves the same final state.

**DELETE is idempotent, not safe.** The first call deletes; subsequent calls change nothing further (often returning `404`). Idempotent means *same end state*, not *same response*.

### Idempotency ≠ safety

- **Safe** = no state change at all (GET, HEAD, OPTIONS).
- **Idempotent** = repeating the request leaves the same end state (GET, PUT, DELETE, HEAD).
- All safe methods are idempotent; the reverse is not true.

This is the reason browsers happily prefetch and retry GETs, but warn you before re-submitting a POST on refresh.

### The security caveat

POST is **not encrypted**. The body is plaintext on the wire exactly like a URL is. What POST actually buys you is that the data stays out of:

- browser history and bookmarks
- server access logs
- the `Referer` header sent to third-party sites
- shoulder-surfing and copy-pasted URLs

Actual confidentiality comes from **HTTPS**, which encrypts the URL path, query string, headers, and body alike. "POST is more secure" is shorthand for "POST doesn't leak into places that persist," not for encryption.

### HTML forms only speak GET and POST

Native `<form>` elements support just those two methods. To hit PUT/PATCH/DELETE from a browser form, either use `fetch()` from JavaScript or use a method-override trick:

```html
<!-- with the method-override middleware -->
<form action="/products/42?_method=DELETE" method="POST">
  <button>Delete</button>
</form>
```

```js
const methodOverride = require('method-override');
app.use(methodOverride('_method'));
```

APIs called from JS don't need this — `fetch('/products/42', { method: 'DELETE' })` works directly.

### Common status codes

| Code | Meaning | When |
|---|---|---|
| 200 | OK | Successful GET, PUT, PATCH |
| 201 | Created | Successful POST |
| 204 | No Content | Successful DELETE with empty body |
| 400 | Bad Request | Malformed or invalid payload |
| 401 / 403 | Unauthorized / Forbidden | Not logged in / logged in but not permitted |
| 404 | Not Found | Resource doesn't exist |
| 422 | Unprocessable Entity | Well-formed but fails validation |
| 500 | Internal Server Error | Unhandled server-side failure |

### Where each kind of parameter lives in Express

| Source | Express accessor | Example |
|---|---|---|
| Path segment | `req.params` | `/products/:id` → `req.params.id` |
| Query string | `req.query` | `/products?sort=price` → `req.query.sort` |
| Request body | `req.body` | POST JSON payload (needs a body parser) |
| Headers | `req.headers` | `Authorization`, `Content-Type` |

`req.body` is `undefined` until you register a body-parsing middleware — the single most common cause of "why is my POST data empty."

### The "POST for everything" anti-pattern

It's tempting to make every route a POST. Doing so throws away real infrastructure benefits: CDN and browser caching, safe retries, prefetching, bookmarkable URLs, and readable logs. Keep reads as GETs.

The mirror-image mistake is worse: using GET to mutate data (`GET /products/42/delete`). Any crawler, link prefetcher, or antivirus scanner that follows the link will silently execute the mutation.

### RESTful route table (the seven conventional routes)

| Name | Verb | Path | Purpose |
|---|---|---|---|
| Index | GET | `/products` | List all |
| New | GET | `/products/new` | Serve the creation form |
| Create | POST | `/products` | Persist the new item |
| Show | GET | `/products/:id` | Display one |
| Edit | GET | `/products/:id/edit` | Serve the edit form |
| Update | PUT/PATCH | `/products/:id` | Persist the changes |
| Destroy | DELETE | `/products/:id` | Remove it |

Note that `new` and `edit` are GETs — they only *render forms*. The mutation happens on the subsequent POST/PUT/DELETE to the bare resource path.

### Route ordering gotcha

Express matches routes top-down, first match wins:

```js
app.get('/products/new', ...);   // must come FIRST
app.get('/products/:id', ...);   // otherwise ":id" captures "new"
```

Place literal/static segments above parameterized ones.

---

## See Also

- [[HTTP Status Codes]]
- [[Express Middleware]]
- [[REST API Design]]